# Track 5 starter: Incident triage and logistics analysis (synthetic)
Summaries to support **observers and civic organisations**. Protect reporters: no real names, minimal location detail, no public exposure of unverified accusations.

In [ ]:
import sys; sys.path.append('../src')
import pandas as pd, numpy as np
from ai4elections import load_csv, load_jsonl
pd.set_option('display.max_columns', 30)

In [ ]:
inc = load_csv('incident_reports_synthetic.csv')
inc.groupby('category')['severity_0_4'].agg(['count','mean']).round(2).sort_values('mean', ascending=False)

In [ ]:
# Baseline text classifier for incident category
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score
cl = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=500))
print(cross_val_score(cl, inc['description'], inc['category'], cv=5).mean().round(3))

In [ ]:
# Logistics: what drives late delivery?
lg = load_csv('logistics_deliveries_synthetic.csv')
print(lg.groupby('settlement')['transit_hours'].describe().round(2))
from sklearn.ensemble import GradientBoostingRegressor
X = pd.get_dummies(lg[['distance_km','settlement','vehicle','dispatch_hour']], drop_first=True)
model = GradientBoostingRegressor(random_state=0).fit(X, lg['transit_hours'])
pd.Series(model.feature_importances_, X.columns).sort_values(ascending=False).round(3)

## Your turn
- Build a severity-ranked triage queue with a human review step.
- Join incidents to `polling_units_synthetic.csv` to find accessibility gaps by state.
- Add data-minimisation and consent-by-design to your reporting flow.